In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from sqlalchemy.orm import joinedload

import src
import src.data.doccano_models as dm
import src.data.models as m

In [3]:
pd.set_option("display.max_rows", 256)
doccano_engine = create_engine(src.DOCCANO_ENGINE)
ps_engine = create_engine(src.PS_ENGINE)
doccano_session = Session(doccano_engine)
ps_session = Session(ps_engine)

In [4]:
PROJECT = "YT_Pop_1"

In [5]:
doccano_examples = (
    doccano_session.query(dm.ExamplesExample)
    .options(
        joinedload(dm.ExamplesExample.labels),
        joinedload(dm.ExamplesExample.state),
    )
    .join(dm.ExamplesExample.project)
    .join(dm.ExamplesExample.state)
    .filter(
        dm.ProjectsProject.name == PROJECT,
        dm.ExamplesExample.state.any(),
    )
)

In [6]:
def get_video(s, video_id):
    video = s.query(m.Video).filter(m.Video.id == video_id).one()
    return video

In [216]:
rows = []
for ex in doccano_examples:
    row = {}
    try:
        assert len(ex.labels) == 1
    except AssertionError:
        continue

    row["label"] = str(ex.labels[0])
    if row["label"].startswith("Rede"):
        row["label"] = "Rede"
    elif row["label"] == "Erklärvideo":
        row["label"] = "Teaser / Clip"

    video_id = ex.meta["id"]
    row["video_id"] = video_id
    row["channel"] = ex.meta["uploader_id"]
    video = get_video(ps_session, video_id)
    row["duration"] = video.duration
    row["title"] = video.title
    if "podcast" in video.title:
        row["title_podcast"] = True
    else:
        row["title_podcast"] = False
    row["description"] = video.description
    sents = sorted(video.sentences, key=lambda x: x.sentence_no, reverse=False)
    row["video_intro"] = " ".join(
        str(tok) for sent in sents[:15] for tok in sent.tokens
    )
    rows.append(row)

In [217]:
df = pd.DataFrame(rows)

channel_encoder = LabelEncoder()
df.channel = channel_encoder.fit_transform(df.channel.astype(str))

In [218]:
df.label.value_counts()

label
Rede                342
Teaser / Clip        73
Interview            23
Podcast              20
Disskusionsrunde     19
Ausschusssitzung      1
Name: count, dtype: int64

In [219]:
# filter all labels that occur only once
df = df[df.groupby("label").label.transform(len) > 1]

le = LabelEncoder()
df.label = df.label.astype(str)
le.fit(df.label)


train, test = train_test_split(
    df,
    stratify=df["label"],
    random_state=1337,
    test_size=0.4,
)

In [220]:
train.shape, test.shape

((286, 8), (191, 8))

In [221]:
cols = [
    "title",
    "description",
    "video_intro",
    "channel",
    "duration",
]

X_train = train[cols]
X_test = test[cols]

y_train = le.transform(train["label"])
y_true = le.transform(test["label"])

In [222]:
from imblearn.ensemble import RUSBoostClassifier
from sklearn.preprocessing import MinMaxScaler

# class IdentityTransformer(BaseEstimator, TransformerMixin):
#     def __init__(self):
#         pass
#
#     def fit(self, input_array, y=None):
#         return self
#
#     def transform(self, input_array, y=None):
#         return input_array*1

column_transformer = ColumnTransformer(
    remainder="passthrough",
    transformers=[
        ("std_duration", MinMaxScaler(), ["duration"]),
        # ("channel", IdentityTransformer(),["channel"]),
        ("title_vectorizer", TfidfVectorizer(use_idf=True, analyzer="char"), "title"),
        (
            "desc_vectorizer",
            TfidfVectorizer(use_idf=True, analyzer="char"),
            "description",
        ),
        (
            "intro_vectorizer",
            TfidfVectorizer(use_idf=True, analyzer="char"),
            "video_intro",
        ),
    ],
)

pipeline = Pipeline(
    [
        ("preprocessing", column_transformer),
        (
            "classifier",
            RUSBoostClassifier(n_estimators=500, learning_rate=1, replacement=False),
        ),
    ],
)

In [223]:
_ = pipeline.fit(X_train, y_train)

In [224]:
y_test = pipeline.predict(X_test)

In [225]:
print(confusion_matrix(y_true, y_test))

[[  1   0   0   7   0]
 [  0   0   0   9   0]
 [  0   0   1   7   0]
 [  0   1   0 120  16]
 [  0   0   0  16  13]]


In [227]:
print(classification_report(y_true, y_test, zero_division=0, target_names=le.classes_))

                  precision    recall  f1-score   support

Disskusionsrunde       1.00      0.12      0.22         8
       Interview       0.00      0.00      0.00         9
         Podcast       1.00      0.12      0.22         8
            Rede       0.75      0.88      0.81       137
   Teaser / Clip       0.45      0.45      0.45        29

        accuracy                           0.71       191
       macro avg       0.64      0.31      0.34       191
    weighted avg       0.69      0.71      0.67       191



In [231]:
feats = pipeline["classifier"].fea
imps = pipeline["classifier"].feature_importances_

array([0.14 , 0.012, 0.002, 0.   , 0.002, 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.004, 0.004,
       0.   , 0.   , 0.   , 0.   , 0.008, 0.   , 0.   , 0.004, 0.   ,
       0.004, 0.01 , 0.014, 0.004, 0.014, 0.018, 0.012, 0.006, 0.002,
       0.   , 0.008, 0.006, 0.002, 0.006, 0.01 , 0.   , 0.   , 0.006,
       0.   , 0.008, 0.   , 0.002, 0.016, 0.   , 0.   , 0.006, 0.004,
       0.   , 0.   , 0.   , 0.002, 0.   , 0.   , 0.002, 0.002, 0.   ,
       0.002, 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.02 , 0.036,
       0.008, 0.   , 0.002, 0.   , 0.   , 0.   , 0.   , 0.   , 0.002,
       0.   , 0.01 , 0.   , 0.014, 0.008, 0.   , 0.01 , 0.002, 0.   ,
       0.002, 0.   , 0.002, 0.   , 0.002, 0.   , 0.008, 0.   , 0.002,
       0.   , 0.012, 0.008, 0.006, 0.004, 0.016, 0.006, 0.016, 0.012,
       0.004, 0.006, 0.002, 0.008, 0.006, 0.004, 0.008, 0.01 , 0.026,
       0.   , 0.004, 0.006, 0.016, 0.004, 0.   , 0.008, 0.   , 0.004,
       0.01 , 0.   ,

In [233]:
pipeline["classifier"].feature_names = list(X_train.columns.values)

In [236]:
cl = pipeline["classifier"]
cl.feature_importances_

array([0.14 , 0.012, 0.002, 0.   , 0.002, 0.   , 0.   , 0.   , 0.   ,
       0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.004, 0.004,
       0.   , 0.   , 0.   , 0.   , 0.008, 0.   , 0.   , 0.004, 0.   ,
       0.004, 0.01 , 0.014, 0.004, 0.014, 0.018, 0.012, 0.006, 0.002,
       0.   , 0.008, 0.006, 0.002, 0.006, 0.01 , 0.   , 0.   , 0.006,
       0.   , 0.008, 0.   , 0.002, 0.016, 0.   , 0.   , 0.006, 0.004,
       0.   , 0.   , 0.   , 0.002, 0.   , 0.   , 0.002, 0.002, 0.   ,
       0.002, 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.02 , 0.036,
       0.008, 0.   , 0.002, 0.   , 0.   , 0.   , 0.   , 0.   , 0.002,
       0.   , 0.01 , 0.   , 0.014, 0.008, 0.   , 0.01 , 0.002, 0.   ,
       0.002, 0.   , 0.002, 0.   , 0.002, 0.   , 0.008, 0.   , 0.002,
       0.   , 0.012, 0.008, 0.006, 0.004, 0.016, 0.006, 0.016, 0.012,
       0.004, 0.006, 0.002, 0.008, 0.006, 0.004, 0.008, 0.01 , 0.026,
       0.   , 0.004, 0.006, 0.016, 0.004, 0.   , 0.008, 0.   , 0.004,
       0.01 , 0.   ,

In [119]:
predicted_labels = le.inverse_transform(y_test)

In [120]:
test["y_test"] = predicted_labels
test.channel = channel_encoder.inverse_transform(test.channel)

In [121]:
test

,label,video_id,channel,duration,title,description,video_intro,y_test
280,Rede,pdlIAFuKRQI,@FDP,421,Zur Wahl des Ministerpräsidenten in #Thüringen.,Der FDP-Bundesvorsitzende Christian Lindner im...,Herr Präsident ! Meine Kolleginnen und Kollege...,Interview
77,Rede,7nJ4egPFucM,@AfDFraktionimBundestag,3595,"Anträge der AfD-Fraktion zum ""Globalen Pakt fü...",Folge uns auch auf Telegram: https://t.me/afdf...,"Ja , meine Damen und Herren , lassen Sie uns b...",Podcast
118,Rede,svChDSaiQGA,@AfDTV,220,Klare Worte zu anhaltenden Corona-Gängeleien d...,++💙 Gesund ohne Zwang Aktionstag 2022 💙++\nA...,"Sehr geehrte Frau Präsidentin , sehr geehrte D...",Erklärvideo
307,Teaser / Clip,l0k7JWsRFi0,@cdutv,30,#CDUVorsitz: Teaser CDU.TV-Interviews,,Warum ich für den Parteivorsitz der CDU Deutsc...,Teaser / Clip
242,Podcast,GtshKaJOhB8,@FDP,94,Christian Lindner & Annabel Oelmann über die B...,Finanzielle Bildung ist der Schlüssel zu einem...,"Und ich glaube ja , wenn man über Finanzen red...",Rede
416,Teaser / Clip,6p0dY2nx8cU,@csumedia,47,Einladung für LeFloid,"Lieber LeFloid, wir haben in Deinem neuesten V...","Hey LeFloid , ich bin hier in der Hochschule f...",Teaser / Clip
353,Erklärvideo,uu8-TVfivtY,@cdutv,30,Sicherheit,"Gemeinsam für ein Land, in dem alle in Sicherh...",Der die Sicherheit seiner Bürger garantiert . ...,Erklärvideo
188,Rede,WnXV3ZmC_mI,@DIELINKE,429,Diese Groko wird die zentralen sozialen Proble...,Statement von Bernd Riexinger zu den Ergebniss...,"Guten Morgen . Ich gehe davon aus , dass die E...",Interview
14,Interview,PsnQ9ERQKMs,@AfDTV,674,Beatrix von Storch | AfD persönlich,Die Bundestagsabgeordnete Beatrix von Storch i...,Hallo und herzlich willkommen zu einer neuen F...,Interview
360,Rede,NDk8HXK6yO8,@cdutv,2136,25. Politischer Aschermittwoch aus Demmin: Red...,Zum 25. Mal findet der Politische Aschermittwo...,"Lieber Werner Kuhn , lieber Thorsten Renz , di...",Podcast
